In [ ]:
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

model_id = "Arthur-75/storm-qwen3-8B"

system_prompt = (
    "From the query generate new semantic related keywords.\n"
    "Output the result strictly as a single comma-separated line."
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
#tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    #device_map=None,
    #low_cpu_mem_usage=True,
    trust_remote_code=True,
)

model = model.to(device)
model.eval()



def generate_keywords(query: str):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"[QUERY]: {query.strip()}\n[KEYWORDS]: "},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=64,#[32,64]
            do_sample=False,
            num_beams=6,
            num_beam_groups=3,
            diversity_penalty=1.0,
            num_return_sequences=3,
            #repetition_penalty=1.0,
            #custom_generate="transformers-community/group-beam-search"
            custom_generate="storm/eval/community_tools"

           # pad_token_id=tokenizer.pad_token_id,
           # eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(
        output[:, prompt_len:],
        skip_special_tokens=True,
    )

    return decoded


query = "What are the symptoms of vitamin D deficiency?"
outputs = generate_keywords(query)
outputs = " ".join(outputs)

print(outputs)


/opt/anaconda3/envs/GA/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[' Vitamin deficiency symptoms include fatigue D symptoms dry skin symptoms deficiency can include nausea vitamin D symptoms headache deficiency symptoms muscle weakness vitamin D symptoms fatigue can include dry skin vitamin D symptoms headache can include muscle weakness vitamin D symptoms fatigue can include dry skin vitamin D symptoms headache can include muscle weakness vitamin D symptoms fatigue can include dry skin vitamin D', ' Vitamin deficiency symptoms include fatigue D symptoms dry skin symptoms deficiency can include nausea vitamin D symptoms weakness symptoms deficiency can include headache vitamin D symptoms weakness symptoms deficiency can include nausea vitamin D symptoms weakness symptoms deficiency can include nausea vitamin D symptoms weakness symptoms deficiency can include nausea vitamin D symptoms weakness symptoms deficiency can include nausea vitamin D symptoms weakness symptoms', ' Vitamin D deficiency symptoms include fatigue symptoms vitamin D symptoms nause